# Compute fitness effects for subset trees

Use expected rates from the global neutral model to compute fitness effects for each subset (host, geographic, split-half).

In [1]:
import pandas as pd
import numpy as np

## Read data

In [2]:
# Read subset counts
counts_df = pd.read_csv('../results/subset_counts.csv', keep_default_na=False, low_memory=False)
counts_df = counts_df.replace('', np.nan)
counts_df['codon_site'] = counts_df['codon_site'].astype(str)
print(f'Subset counts: {len(counts_df):,} rows')
print(f'Subsets: {sorted(counts_df["subset"].unique())}')

Subset counts: 4,357,290 rows
Subsets: ['asia', 'avian', 'europe', 'human', 'north_america', 'split_a', 'split_b', 'swine']


In [3]:
# Read expected rates from the global neutral model
predicted_rates_df = pd.read_csv('../results/expected_rates.csv', keep_default_na=False)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200380
1,AC,HA,AAC,0.315456
2,AC,HA,AAG,0.246664
3,AC,HA,AAT,0.308056
4,AC,HA,CAA,0.178563


## Compute expected counts and filter

In [4]:
counts_cutoff = 1

actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    .query('actual_count >= @counts_cutoff | expected_count >= @counts_cutoff')
)
print(f'Rows after filtering (>= {counts_cutoff} actual or expected): {len(actual_expected_df):,}')
actual_expected_df.head()

Rows after filtering (>= 1 actual or expected): 873,237


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,subset,subset_type,evo_opp,rate,predicted_rate,expected_count
3,312,A312C,A,C,PA,3,104,AAA,AAC,K,...,all,PA,PA_all,2148,north_america,geographic,15.811918,0.12648686844894594,0.311752,4.929401
22,765,A765C,A,C,PB2,3,255,GTA,GTC,V,...,all,PB2,PB2_all,2277,split_b,split_half,2.983311,0.3351979979390549,0.154513,0.460962
38,767,A767C,A,C,PB2,2,256,GAC,GCC,D,...,all,PB2,PB2_all,2277,split_b,split_half,36.890646,0.0,0.132380,4.883571
46,311,A311C,A,C,PA,2,104,AAA,ACA,K,...,all,PA,PA_all,2148,north_america,geographic,15.923650,0.0,0.202785,3.229072
49,311,A311C,A,C,PA,2,104,AAG,ACG,K,...,all,PA,PA_all,2148,north_america,geographic,14.703911,0.20402735562310031,0.249623,3.670442


## Compute amino-acid-level fitness effects

In [5]:
pseudo_count = 0.5

groupby_cols = [
    'subset', 'subset_type', 'subtype', 'segment', 'gene',
    'codon_site', 'wt_aa', 'mut_aa', 'aa_mut', 'mut_class'
]

aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x:
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)

aa_fitness_df.to_csv('../results/subset_aa_fitness_effects.csv', index=False)
print(f'AA-level fitness effects: {len(aa_fitness_df):,} rows')
aa_fitness_df.head()

AA-level fitness effects: 456,160 rows


,subset,subset_type,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,asia,geographic,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,38.084507,-4.345998
1,asia,geographic,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,4.620926,-2.326483
2,asia,geographic,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,2.936988,-1.927743
3,asia,geographic,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,13.846674,-3.356665
4,asia,geographic,H1,HA,HA,10,C,S,C10S,nonsynonymous,1,0.085591,0.940599


In [6]:
# Summary by subset and mutation class
aa_fitness_df.groupby(['subset', 'mut_class']).size().unstack(fill_value=0)

mut_class,nonsense,nonsynonymous,synonymous
subset,,,
asia,2965,51021,10275
avian,2517,45121,9799
europe,2479,40970,8291
human,2341,40304,7385
north_america,2713,46892,9517
split_a,3010,54663,10959
split_b,3014,54618,11005
swine,1614,28132,6555
